# 2.1 AWS Site, Settings, Storage, And Results

For most users the cloud site is the normal execution path because the fast solver is installed and maintained there. The Python job authoring API is the same as local and HPC sites; the site object handles authentication, upload, remote execution, result metadata, trace fetching, logs, and output files.

By the end, you should be able to author jobs locally, submit them to AWS, fetch logs/results, and analyze traces through the same result API used by local runs.


## How To Read This Tutorial

This notebook is about operational separation: author the same simulation and jobs locally, then let the AWS site decide where storage, execution, logs, and downloads live. Read the cells as a production workflow rather than a cloud demo: build a small but complete job, submit it strictly, fetch the artifacts, and verify the same trace/output APIs work after the data crosses a remote boundary.

The key idea is that a site is infrastructure, not modeling syntax. If a job changes meaning when it moves from local to AWS, that is a bug or a configuration mismatch; the project, simulation, acquisition, and output requests should remain ordinary FrequenSolve objects.

## Configuration Checklist

AWS is the managed path most users should start with: the fast solver, storage, logs, and job lifecycle are all handled by the cloud site. The Python objects stay deliberately consistent across sites: a `Project` owns simulations and jobs, a `Job` describes work, and a `Site` decides where that work runs.

| Concern | AWS site | HPC site | Local site |
| --- | --- | --- | --- |
| Solver availability | Managed in the cloud environment. | Must be installed on the target cluster. | Must be installed on this workstation. |
| Staging | Uploads project/job artifacts to cloud storage. | Copies or syncs artifacts to a remote work directory. | Writes artifacts directly under the project path. |
| Execution | Submitted through the cloud service. | Submitted through the scheduler, usually SLURM. | Runs a local worker process. |
| Results | Fetched from cloud storage through the result API. | Fetched from the remote work directory. | Read directly from local result directories. |
| Best first use | Production and normal user workflows. | Institution-managed clusters with solver access. | API development, smoke tests, and small examples. |

The run cells below are intentionally strict. If AWS authentication, project configuration, storage access, or cloud solver availability is wrong, the cell should fail with enough site/job context to inspect logs rather than hiding the problem behind a broad exception handler.


## Site Mental Model

A site changes where a completed job runs; it should not change how the simulation is authored or how results are read. Keep the project, simulation, acquisition, jobs, trace reads, and ParaView output requests ordinary. Let the site object handle authentication, staging, scheduling, storage, polling, and fetching.

That separation is what makes it possible to prototype locally, run production jobs in the cloud or on HPC, and keep the analysis cells nearly identical.


## Run Handles, Futures, And Task Summaries

`site.submit(job)` returns a run handle rather than raw arrays. The handle represents work that may still be queued, running, packing traces, or fetching results. Calling `.wait()` blocks until the site reaches a terminal state and returns a result object with logs, metadata, output files, and trace accessors.

For frequency sweeps, a completed job may still report some failed tasks. Always read the printed task summary and the run manifest before interpreting traces: `succeeded` means the task completed and converged, `complete` means a task produced a terminal result, and `failed` identifies frequencies that need log review.

## Imports And Shared Job Builder

The helper creates one simulation and two jobs: a time-domain trace job and a single-frequency ParaView QC job. That pair is repeated across all site tutorials so the only moving part is the `Site` object.

The builder returns project-owned jobs rather than loose JSON. Calling `site.submit(job)` serializes the simulation, job, acquisition, mesh, outputs, and units into the project structure before staging or execution. That is the core site contract users should remember: author locally, submit through the selected site, fetch through the returned result handle.


In [ ]:
import os

import numpy as np
import frequensolve as fs

u = fs.ureg


In [ ]:
def build_acoustic_tutorial_jobs(project_path, *, simulation_name, trace_job_name, qc_job_name, f_max=25.0):
    project = fs.Project(
        name="project",
        pretty_name=simulation_name,
        path=project_path,
        log_level="INFO",
        log_to_console=True,
    )
    sim = project.new_simulation(
        name=simulation_name,
        physics="acoustic",
        dimension=2,
        units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
    )

    model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
    model.add_surface(name="top", depth=0.0 * u.km)
    model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
    model.add_surface(name="interface", depth=0.22 * u.km)
    model.add_layer(name="basement", properties={"Vp": 2.4 * u.km / u.s, "Rho": 2.2 * u.g / u.cm**3})
    model.add_surface(name="bottom", depth=0.5 * u.km)
    sim += model

    sim += model.hex_mesh_generator([8, 4])
    sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=f_max)
    sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
    sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

    acq = fs.Acquisition()
    acq.add_source_group(kind="scalar", coords=[[0.35, 0.05], [0.65, 0.05]])
    hydrophone = fs.ReceiverNode(name="hydrophone")
    hydrophone.add_component(name="p", field="pressure")
    acq.add_receiver_group(name="surface", device=hydrophone, coords=[[x, 0.04] for x in np.linspace(0.1, 0.9, 61)])
    sim += acq
    sim += fs.Discretization()
    sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

    trace_job = fs.TimeDomainJob(
        name=trace_job_name,
        simulation=sim,
        f_min=0.0,
        f_max=f_max,
        T_max=0.9,
    )
    qc_job = fs.FrequencyDomainJob(
        name=qc_job_name,
        simulation=sim,
        f_list=[12.0],
        outputs=[
            fs.ParaviewOutput(
                name="pv_qc",
                fields=["pressure"],
                properties=["vp", "rho", "Subdomain"],
                show_pml=True,
                upscale=0,
                order=1,
            )
        ],
    )
    return project, sim, trace_job, qc_job


## Author The Jobs Locally

The trace job is time-domain and produces receiver data. The ParaView QC job is a separate single-frequency job because VTK output is frequency-domain by design. Keeping them separate makes the site workflow explicit: submit the data job, submit the QC output job, then fetch logs and artifacts for each.

The QC job uses `upscale=0` to keep cloud visualization artifacts small while still proving that mesh, PML, and material outputs were produced.


In [ ]:
project, sim, trace_job, qc_job = build_acoustic_tutorial_jobs(
    "./scratch/tutorials/aws_site",
    simulation_name="aws_site_acoustic",
    trace_job_name="time_aws_site",
    qc_job_name="freq_aws_site_qc",
)
project.save()
{
    "trace_job": trace_job.to_fs(),
    "qc_job": qc_job.to_fs(),
}


## Configure And Submit To AWS

`AWSSite` handles remote staging, job launch, polling, and result download. Most users will run production jobs on AWS because a runnable site must have the fast solver installed and configured.


In [ ]:
domain = os.environ.get("FREQUENSOL_DOMAIN")
site = fs.AWSSite(domain=domain, interactive=True, verbose=True)
trace_result = site.submit(trace_job).wait()
qc_result = site.submit(qc_job).wait()
{
    "trace_status": trace_result.status,
    "qc_status": qc_result.status,
}


## Fetch Logs, Traces, And Output Files

The result API is site-independent. On AWS these calls fetch from cloud storage; on local they read from disk; on HPC they fetch/stage from the remote work directory.


In [ ]:
logs = trace_result.logs()
qc_outputs = qc_result.output_files(existing=True)
traces = trace_result.traces(upscale=4)
{
    "trace_successful": trace_result.successful,
    "qc_successful": qc_result.successful,
    "logs": str(logs),
    "qc_output_count": len(qc_outputs),
    "trace_files": traces.files,
}


## Plot A Retrieved Trace Gather

This plot demonstrates that cloud results are consumed through the same `TraceDataset` workflow used elsewhere in the tutorials.


In [ ]:
traces = trace_result.traces(upscale=4)
wavelet = fs.RickerWavelet(f=12.0)
group = traces.groups[0]
component = traces.components(group)[0]
source = traces.sources(group)[0]
gather = traces.td(group, component, source, wavelet, upscale=4, T_max=0.9)
fs.plot_gather(
    gather,
    A=2.0 * np.nanstd(np.real(gather.values)),
    cmap="gray",
    figsize=(9, 4),
    title=f"{trace_job.name}: {group}/{component}/source {source}",
)


## Before Moving On

A user should leave this notebook understanding which parts are cloud-specific and which parts are universal. Cloud-specific concepts are authentication, storage location, launch settings, and fetched logs. Universal concepts are project-owned simulations, job-owned outputs, result handles, `TraceDataset`, and ParaView file discovery.

For a production AWS run, inspect the job contract and logs before interpreting traces. The cloud site makes large runs accessible, but it does not remove the need to confirm frequencies, receiver groups, output requests, and solver convergence summaries.

## Result Review Checklist

The AWS-specific part of this notebook is site configuration and remote storage. The simulation, jobs, trace reads, and output discovery should look the same as the local and HPC notebooks.

| Artifact | What to confirm |
| --- | --- |
| Site status output | Authentication, upload, launch, polling, and download all complete for both jobs. |
| `trace_result.logs()` | Logs are fetched from cloud storage and point to the remote solver run, not a local placeholder. |
| `trace_result.traces()` | Trace files open through the same `TraceDataset` API used locally. |
| `qc_result.output_files(...)` | ParaView files are discoverable without hard-coding cloud storage paths. |

For production AWS runs, keep the project path stable and use job names that encode the model version, frequency band, and output purpose. That makes cloud result browsing much less mysterious later.
